# KuaiRand-27K: residual/connection variant comparison

dlrm_v3 HSTU on KuaiRand-27K, 10 seeds x 1 epoch, end-of-epoch eval.
Five variants compared: STU, NeuTRENO, NeuTRENO_AN, AttnRes, mHC.

Per-task metrics (mean and std over 10 seeds) are read from `logs/kr27k_results_10seeds.csv`.

In [ ]:
import os, re, csv
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio

# Bento-Plotly registered template (sets pio.templates.default + add_source + OKABE_ITO)
exec(open(os.path.expanduser(
    "/home/ngocbh/.claude/agent-market/plugins/10x-data-scientist/skills/visualization/bento-plotly/references/bento_style_template.py"
)).read())

In [ ]:
# --- Load per-method mean and std from the results CSV ---
CSV = "/storage/home/ngocbh/project/gr/logs/kr27k_results_10seeds.csv"
METHODS = ["STU", "NeuTRENO", "NeuTRENO_AN", "AttnRes", "mHC"]

rows = []
with open(CSV) as fh:
    r = csv.reader(fh)
    next(r)
    last_task = None
    for line in r:
        task = line[0] or last_task
        last_task = task
        metric = "NE" if "NE" in line[1] else "GAUC"
        for i, m in enumerate(METHODS):
            mm = re.search(r"([\d.]+) \\pm ([\d.]+)", line[2 + i])
            rows.append(dict(task=task, metric=metric, method=m,
                             mean=float(mm.group(1)), std=float(mm.group(2))))

df = pd.DataFrame(rows)
TASK_ORDER = ["is_click", "long_view", "is_profile_enter", "is_like",
              "is_comment", "is_follow", "is_forward", "is_hate"]
print(df[df.metric == "NE"].pivot_table(index="task", columns="method",
      values="mean").reindex(TASK_ORDER)[METHODS].round(4))

In [ ]:
def comparison_fig(metric, ylabel, title):
    """Grouped point plot (mean + std error bar) across tasks, one series per variant."""
    sub_df = df[df.metric == metric]
    tasks = TASK_ORDER
    n = len(METHODS)
    fig = go.Figure()
    for j, m in enumerate(METHODS):
        d = sub_df[sub_df.method == m].set_index("task")
        offset = (j - (n - 1) / 2) * 0.14
        xpos = [i + offset for i in range(len(tasks))]
        means = [d.loc[t, "mean"] for t in tasks]
        stds = [d.loc[t, "std"] for t in tasks]
        fig.add_trace(go.Scatter(
            x=xpos, y=means, mode="markers", name=m, text=tasks, customdata=stds,
            error_y=dict(type="data", array=stds, visible=True, thickness=1.2, width=3),
            marker=dict(color=OKABE_ITO[j], size=9),
            hovertemplate=(f"<b>{m}</b><br>%{{text}}<br>"
                           f"{ylabel}=%{{y:.4f}} (std %{{customdata:.4f}})<extra></extra>")))
    fig.update_layout(
        title=dict(text=f"<b>{title}</b>"),
        xaxis=dict(title="task", tickmode="array", tickvals=list(range(len(tasks))),
                   ticktext=tasks, range=[-0.5, len(tasks) - 0.5]),
        yaxis=dict(title=ylabel),
        legend=dict(orientation="v", yanchor="top", y=1, xanchor="left", x=1.02),
        margin=dict(r=200, b=100))
    add_source(fig, "KuaiRand-27K, dlrm_v3 HSTU, 10 seeds, 1 epoch, end-of-epoch eval", y=-0.24)
    return fig

In [ ]:
# --- NE: lower is better ---
fig_ne = comparison_fig("NE", "eval NE (lower = better)", "Eval NE by task")
fig_ne.show()

In [ ]:
# --- gAUC: higher is better ---
fig_gauc = comparison_fig("GAUC", "eval gAUC (higher = better)", "Eval gAUC by task")
fig_gauc.show()

In [ ]:
# --- Optional: export static PNGs for sharing ---
OUT = "/storage/home/ngocbh/project/gr/logs/mhc_analysis"
os.makedirs(OUT, exist_ok=True)
try:
    fig_ne.write_image(f"{OUT}/kr27k_variants_ne.png", scale=2)
    fig_gauc.write_image(f"{OUT}/kr27k_variants_gauc.png", scale=2)
    print("wrote PNGs to", OUT)
except Exception as e:
    print("PNG export skipped (needs kaleido):", e)